# 頭痛BERT: 順次BERT vs 選択強化BERT

## 概要
頭痛プロトコル（`protocol.yaml` の `headache`）の遷移図に対して、2種類のBERTモデルを実装・比較する。

- **モデル1：全BERT（順次）**
  yaml の分岐ノード数だけ専用BERTを用意し、「**分岐 → 選択肢 → 次の分岐**」のサイクルに従って順次（greedy）に推論する。
  予測が「次へ進む」以外の場合はその時点で終端し、後続のBERTは呼ばない。

- **モデル2：選択強化BERT（単一 MultipleChoice BERT）**
  分岐質問・関連会話ペア・選択肢を入力に取る単一のBERTで、選択肢を予測する。
  yaml の同じ「**分岐 → 選択肢**」サイクルに従って遷移図をたどる。

データ前処理（ラリー分割・Sentence-LUKE ベクトル化・cos類似度による採用ペア選定）と各BERTの学習方法は既存ノートブック (`BERTの入力文選定.ipynb` / `頭痛BERT学習.ipynb`) を踏襲する。

# 1. データ準備

In [ ]:
# Colab で実行する場合のみ Drive をマウント（ローカル実行時は不要）
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
import pandas as pd
# パス区切りはスラッシュ推奨（Windows のバックスラッシュは Python の文字列エスケープと衝突する）
df_all = pd.read_csv(
    'C:/Users/hiyok/Desktop/Emergency_task/dataset/headache_emergency_calls202605311132.csv'
)
# 痛みの各値（0/1/2）から偏らないように 4件ずつ取り、最初の10件を採用する。
# （単に先頭10件を取ると痛み=0 のみに偏り、後段の BERT 学習で各分岐のラベルが
#  単一クラスになり stratified split が失敗するため）
_parts = []
for _v in [0, 1, 2]:
    _sub = df_all[df_all['痛み'] == _v].drop_duplicates(['しびれ', '振る舞い', 'トリアージ'])
    _parts.append(_sub.head(4))
df = pd.concat(_parts).head(10).reset_index(drop=True)
df

## 1.1. 会話をラリーごとに分割し、質問と回答のペアを作成

In [ ]:
import re

def parse_conversation(conversation_text):
    dispatcher_turns = re.findall(r'Dispatcher:([^\n]*)', conversation_text)
    caller_turns = re.findall(r'Caller:([^\n]*)', conversation_text)

    qa_pairs = []
    min_len = min(len(dispatcher_turns), len(caller_turns))

    for i in range(min_len):
        question = dispatcher_turns[i].strip()
        answer = caller_turns[i].strip()
        if question and answer:
            qa_pairs.append({'質問': question, '回答': answer})

    return qa_pairs

df['qa_pairs'] = df['会話'].apply(parse_conversation)
df_expanded = df.explode('qa_pairs')
df_expanded['質問'] = df_expanded['qa_pairs'].apply(lambda x: x['質問'] if isinstance(x, dict) else None)
df_expanded['回答'] = df_expanded['qa_pairs'].apply(lambda x: x['回答'] if isinstance(x, dict) else None)

df_pairs = df_expanded.drop(columns=['会話', 'qa_pairs'])
df_pairs['ペア'] = df_pairs['質問'] + ' ' + df_pairs['回答']
df_pairs['ペア番号'] = df_pairs.groupby('id').cumcount()

ordered_columns = ['id', 'ペア番号', 'ペア', '痛み', 'しびれ', '振る舞い', 'トリアージ', '質問', '回答']
df_pairs = df_pairs[ordered_columns].reset_index(drop=True)
display(df_pairs.head(10))

## 1.2. protocol.yaml から頭痛プロトコルの分岐構造を読み込み

ここで以下を yaml から決定する：
- **BERTの数**（モデル1） = headache プロトコルの分岐ノード数
- **分岐→選択肢のサイクル**（両モデル共通） = 各分岐ノードの選択肢と次ノード/triage

In [ ]:
# %pip install pyyaml  # 実行した場合は Restart Kernel してから次のセルへ
import yaml

with open('C:/Users/hiyok/Desktop/Emergency_task/transition_diagram/protocol.yaml', encoding='utf-8') as f:
    protocol = yaml.safe_load(f)

# headache プロトコルを抽出
headache_proto = next(p for p in protocol['protocols'] if p['id'] == 'headache')

# triage 判定のある choices を持つノード（metadata_only でないもの）を分岐ノードとする
def is_branch_node(node):
    return 'choices' in node and not node.get('metadata_only', False)

branch_nodes = [n for n in headache_proto['nodes'] if is_branch_node(n)]

print(f'頭痛プロトコルの分岐ノード数（= モデル1のBERT数）: {len(branch_nodes)}')
for i, n in enumerate(branch_nodes):
    print(f'\n[分岐 {i+1}] id={n["id"]}')
    print(f'  質問: {n["question"]}')
    for c in n['choices']:
        outcome = c.get('triage') or ('next:' + c.get('next', '')) or 'fallback'
        print(f'    {c["code"]}: {c["text"]}  ->  {outcome}')

In [ ]:
# 分岐 → 選択肢サイクルを構造化（両モデルから参照する遷移テーブル）
fallback_triage = headache_proto['fallback']['if_all_symptom_questions_negative']

def parse_choice(choice):
    code = choice['code']
    text = choice['text']
    triage = choice.get('triage')
    next_id = choice.get('next')
    if triage:
        return {'code': code, 'text': text, 'action': 'terminal', 'triage': triage}
    elif next_id:
        return {'code': code, 'text': text, 'action': 'next', 'next_id': next_id}
    else:
        return {'code': code, 'text': text, 'action': 'fallback', 'triage': fallback_triage}

branch_table = []
for n in branch_nodes:
    branch_table.append({
        'id': n['id'],
        'question': n['question'],
        'choices': [parse_choice(c) for c in n['choices']]
    })

# 表示
import pprint
pprint.pprint(branch_table, sort_dicts=False, width=120)

## 1.3. branch_table 上での着地集計と可視化

ロード済みのデータ（先頭10件）を、`branch_table` の **ground-truth ラベル**（`痛み` / `しびれ` / `振る舞い` カラム）に従って歩かせ、**どの triage に何件着地するか** を集計する。
- 分岐ノード（ひし形）には通過した件数を表示
- 終端 triage（角丸ボックス）には着地件数を表示
- エッジの太さは通過件数に比例（0件のエッジは破線で表示）
- `action='next'` の遷移先が `branch_table` 外（metadata_only ノードなど）の場合は yaml の `fallback.if_all_symptom_questions_negative` で着地扱い

In [ ]:
# %pip install graphviz  # 実行した場合は Restart Kernel してから次のセルへ
from graphviz import Digraph
from collections import defaultdict

# 集計に必要なラベル変換（モデル1セクションで再定義されるが、可視化を前段で完結させるため早期に置く）
branch_to_label_col = {
    'headache_sudden_severe': '痛み',
    'headache_numbness_paralysis': 'しびれ',
    'headache_abnormal_behavior': '振る舞い',
}
LABEL_TO_CHOICE_CODE = {0: 'a', 1: 'b', 2: 'c'}

# ===== 1) 各患者のグラウンドトゥルース・パスを辿って集計 =====
branch_ids = {b['id'] for b in branch_table}
all_patient_ids = sorted(df_pairs['id'].unique())

edge_count = defaultdict(int)        # (src, dst_node_id) -> 通過件数
terminal_count = defaultdict(int)    # triage -> 着地件数
branch_visits = defaultdict(int)     # branch_id -> 通過件数
patient_paths = []                   # (pid, terminal_triage, [(branch, code, dst)])

for pid in all_patient_ids:
    sub = df_pairs[df_pairs['id'] == pid]
    path = []
    current = branch_table[0]['id']
    terminal_triage = None
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        branch_visits[b['id']] += 1
        label_col = branch_to_label_col[b['id']]
        gt_label = int(sub[label_col].iloc[0])
        gt_code = LABEL_TO_CHOICE_CODE[gt_label]
        choice = next(c for c in b['choices'] if c['code'] == gt_code)
        if choice['action'] == 'next':
            target = choice['next_id']
            if target in branch_ids:
                edge_count[(b['id'], target)] += 1
                path.append((b['id'], gt_code, target))
                current = target
                continue
            else:
                # branch_table 外 → fallback で着地
                dst_key = f'TERM_{fallback_triage}'
                edge_count[(b['id'], dst_key)] += 1
                terminal_count[fallback_triage] += 1
                terminal_triage = fallback_triage
                path.append((b['id'], gt_code + '(→fb)', fallback_triage))
                break
        else:  # terminal / fallback
            terminal_triage = choice['triage']
            dst_key = f'TERM_{terminal_triage}'
            edge_count[(b['id'], dst_key)] += 1
            terminal_count[terminal_triage] += 1
            path.append((b['id'], gt_code, terminal_triage))
            break
    patient_paths.append((pid, terminal_triage, path))

# ===== 2) 集計サマリ =====
total = len(all_patient_ids)
print(f'対象患者数: {total}')
print('\n=== 着地 triage の集計 ===')
for t in sorted(terminal_count.keys()):
    print(f'  {t}: {terminal_count[t]} 件')

print('\n=== 各患者のパス ===')
for pid, t, path in patient_paths:
    arrow = ' → '.join([f"{src}/{code}" for src, code, _ in path] + [t])
    print(f'  {pid}: {arrow}')

# ===== 3) Graphviz で集計付き描画 =====
triage_fill = {
    'R1': '#ff6b6b', 'R2': '#ff8787', 'R3': '#ffa94d',
    'Y1': '#ffd43b', 'Y2': '#ffe066',
    'G':  '#a3e635',
}

# 全候補 terminal を列挙（0件のものも描画する）
all_terminals = set()
for b in branch_table:
    for c in b['choices']:
        if c['action'] in ('terminal', 'fallback'):
            all_terminals.add(c['triage'])
        elif c['action'] == 'next' and c['next_id'] not in branch_ids:
            all_terminals.add(fallback_triage)

g = Digraph('headache_branch_table_counts', format='png')
g.attr(rankdir='TB', fontname='Yu Gothic')
g.attr('node', fontname='Yu Gothic')
g.attr('edge', fontname='Yu Gothic')

# 分岐ノード（通過件数付き）
for b in branch_table:
    vis = branch_visits[b['id']]
    g.node(b['id'], label=f"{b['id']}\n[通過 n={vis}]\n{b['question']}",
           shape='diamond', style='filled', fillcolor='#dbe9ff', fontsize='10')

# 終端ノード（着地件数付き）
for t in sorted(all_terminals):
    cnt = terminal_count.get(t, 0)
    g.node(f'TERM_{t}', label=f'{t}\n着地 n={cnt}',
           shape='box', style='rounded,filled',
           fillcolor=triage_fill.get(t, '#cccccc'),
           fontsize='14', fontname='Yu Gothic Bold')

# エッジ（全構造を描く。通過件数で太さ／実線・破線を切替）
for b in branch_table:
    for c in b['choices']:
        if c['action'] == 'next':
            target = c['next_id']
            if target in branch_ids:
                dst = target
            else:
                dst = f'TERM_{fallback_triage}'
        else:
            dst = f'TERM_{c["triage"]}'
        cnt = edge_count.get((b['id'], dst), 0)
        label = f"{c['code']}: {c['text']}\n(n={cnt})"
        g.edge(
            b['id'], dst, label=label, fontsize='9',
            penwidth=str(1 + cnt * 1.2),
            style='solid' if cnt > 0 else 'dashed',
            color='black' if cnt > 0 else 'gray60',
        )

g  # ノートブック上で表示

# 2. ペアと分岐質問のベクトル化 & cos類似度

Sentence-LUKE で「会話の各ラリー（ペア）」と「yaml の分岐質問」をベクトル化し、cos類似度を計算する。
これにより、各分岐質問にどのラリーが対応しているかを抽出する。

In [ ]:
# === 環境構築：すでに整備済みなら下記の %pip 行はそのままコメントで残す ===
# ※ 一度でも %pip install を走らせた場合は、必ず Restart Kernel してから先に進むこと。
#   そうしないと「MLukeTokenizer requires the SentencePiece library but it was not found」
#   のような Dummy 化エラーが発生する。
# %pip install transformers==4.46.3 sentencepiece fugashi ipadic unidic-lite protobuf tiktoken

In [ ]:
from transformers import MLukeTokenizer, LukeModel
import torch


class SentenceLukeJapanese:
    def __init__(self, model_name_or_path, device=None):
        self.tokenizer = MLukeTokenizer.from_pretrained(model_name_or_path)
        self.model = LukeModel.from_pretrained(model_name_or_path)
        self.model.eval()
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.device = torch.device(device)
        self.model.to(device)

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    @torch.no_grad()
    def encode(self, sentences, batch_size=8):
        all_embeddings = []
        for batch_idx in range(0, len(sentences), batch_size):
            batch = sentences[batch_idx:batch_idx + batch_size]
            encoded_input = self.tokenizer(batch, padding='longest', truncation=True, return_tensors='pt').to(self.device)
            model_output = self.model(**encoded_input)
            sentence_embeddings = self._mean_pooling(model_output, encoded_input['attention_mask']).to('cpu')
            all_embeddings.extend(sentence_embeddings)
        return torch.stack(all_embeddings)


MODEL_NAME = 'sonoisa/sentence-luke-japanese-base-lite'
model_luke = SentenceLukeJapanese(MODEL_NAME)

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

# ペアのベクトル化
df_pairs['ペアのベクトル'] = df_pairs['ペア'].progress_apply(
    lambda x: model_luke.encode([x])[0].tolist() if pd.notna(x) else None
)
display(df_pairs[['id', 'ペア番号', 'ペア', 'ペアのベクトル']].head())

In [ ]:
# 各分岐質問のベクトル化（yaml から取得した branch_table を参照）
branch_question_embeddings = {}
for b in branch_table:
    emb = model_luke.encode([b['question']])[0].tolist()
    branch_question_embeddings[b['id']] = emb
    print(f'埋め込み生成: {b["id"]}  (dim={len(emb)})')

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

conversation_embeddings = np.array(df_pairs['ペアのベクトル'].tolist())

# 分岐ごとに cos 類似度カラムを追加
for b in branch_table:
    emb_np = np.array(branch_question_embeddings[b['id']]).reshape(1, -1)
    df_pairs[f'cos_sim_{b["id"]}'] = cosine_similarity(conversation_embeddings, emb_np).flatten()

cos_cols = [f'cos_sim_{b["id"]}' for b in branch_table]
display(df_pairs[['id', 'ペア番号', 'ペア'] + cos_cols])

# 3. 採用ペアの選定（既存ロジック踏襲）

cos類似度の降順に並べ、隣接する類似度の差が閾値を超えたら以降を不採用とする。

In [ ]:
threshold = 0.02

def select_pairs_by_similarity_gap(group_df, cos_sim_col_name, threshold=0.1):
    sorted_group_indexed = group_df.sort_values(by=cos_sim_col_name, ascending=False)
    sorted_group_temp = sorted_group_indexed.reset_index(drop=True)
    adopted_temp = pd.Series(False, index=range(len(sorted_group_temp)))
    if len(sorted_group_temp) > 0:
        adopted_temp.iloc[0] = True
    if len(sorted_group_temp) <= 1:
        return adopted_temp.set_axis(sorted_group_indexed.index).reindex(group_df.index)
    diffs = sorted_group_temp[cos_sim_col_name].diff() * -1
    for i in range(1, len(diffs)):
        if diffs.iloc[i] > threshold:
            adopted_temp.iloc[i:] = False
            break
        else:
            adopted_temp.iloc[i] = True
    return adopted_temp.set_axis(sorted_group_indexed.index).reindex(group_df.index)


for b in branch_table:
    cos_sim_col = f'cos_sim_{b["id"]}'
    adopted_col = f'採用ペア_{b["id"]}'
    df_pairs[adopted_col] = False
    for pid, group in df_pairs.groupby('id'):
        adoption = select_pairs_by_similarity_gap(group, cos_sim_col, threshold)
        df_pairs.loc[group.index, adopted_col] = adoption

show_cols = ['id', 'ペア番号', 'ペア']
for b in branch_table:
    show_cols.append(f'cos_sim_{b["id"]}')
    show_cols.append(f'採用ペア_{b["id"]}')
display(df_pairs[show_cols])

# 4. モデル1：全BERT（順次実行）

- **BERTの数** = `len(branch_table)`（= yaml から取得した頭痛プロトコルの分岐ノード数）
- 各分岐に対し、その分岐の **採用ペア** を入力としてラベル（はい/いいえ/不明）を予測する独立BERTを学習
- 推論時は branch_table を **頭から順に走査** し、各分岐BERTを呼び出して softmax 信頼度が閾値以上なら採用。「次へ進む（=いいえ）」と判定された場合のみ次の分岐へ。それ以外（はい→triage / 不明→R3）はその時点で終端し、後続BERTは呼ばない。

学習時のラベル付与（既存データに準拠）：
- `痛み` カラム → `headache_sudden_severe`
- `しびれ` カラム → `headache_numbness_paralysis`
- `振る舞い` カラム → `headache_abnormal_behavior`

各カラムの値 0/1/2 は yaml の選択肢 a/b/c（はい/いいえ/不明）に対応する。

In [ ]:
# どの分岐ノードに、データのどの label カラムを使うかのマッピング
branch_to_label_col = {
    'headache_sudden_severe': '痛み',
    'headache_numbness_paralysis': 'しびれ',
    'headache_abnormal_behavior': '振る舞い',
}

# データのラベル 0=はい(a), 1=いいえ(b), 2=不明(c)
LABEL_TO_CHOICE_CODE = {0: 'a', 1: 'b', 2: 'c'}
CHOICE_CODE_TO_LABEL = {'a': 0, 'b': 1, 'c': 2}

## 4.1. 各分岐に対して専用BERTを学習

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertJapaneseTokenizer, BertForSequenceClassification
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score

max_len = 128
epochs = 3
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def train_one_epoch(model, dataloader, optimizer):
    model.train()
    total_loss = 0
    for batch in dataloader:
        b_ids, b_mask, b_labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        out = model(b_ids, attention_mask=b_mask, labels=b_labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += out.loss.item()
    return total_loss / len(dataloader)

def evaluate_model(model, dataloader):
    model.eval()
    val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in dataloader:
            b_ids, b_mask, b_labels = [b.to(device) for b in batch]
            out = model(b_ids, attention_mask=b_mask, labels=b_labels)
            val_loss += out.loss.item()
            preds = torch.argmax(out.logits, dim=1).flatten()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(b_labels.cpu().numpy())
    return (val_loss / len(dataloader),
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, average='weighted', zero_division=0))


per_branch_bert = {}  # branch_id -> {'model', 'tokenizer', 'label_encoder', ...}

for b in branch_table:
    bid = b['id']
    label_col = branch_to_label_col[bid]
    adopted_col = f'採用ペア_{bid}'

    filtered = df_pairs[df_pairs[adopted_col] == True]
    grouped = filtered.groupby('id').agg(
        input_text=('ペア', lambda x: ' '.join(x)),
        label=(label_col, 'first'),
    ).reset_index()

    le = LabelEncoder()
    grouped['encoded'] = le.fit_transform(grouped['label'])
    num_labels = grouped['encoded'].nunique()
    print(f'\n=== [{bid}] label_col={label_col}, num_labels={num_labels} ===')

    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        grouped['input_text'], grouped['encoded'], grouped['id'],
        test_size=1/3, random_state=42,
        stratify=grouped['encoded'] if num_labels > 1 else None
    )

    tokenizer = BertJapaneseTokenizer.from_pretrained('cl-tohoku/bert-base-japanese-whole-word-masking')
    enc_train = tokenizer(X_train.tolist(), padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')
    enc_test  = tokenizer(X_test.tolist(),  padding='max_length', truncation=True, max_length=max_len, return_tensors='pt')

    train_ds = TensorDataset(enc_train['input_ids'], enc_train['attention_mask'], torch.tensor(y_train.tolist()))
    test_ds  = TensorDataset(enc_test['input_ids'],  enc_test['attention_mask'],  torch.tensor(y_test.tolist()))
    train_dl = DataLoader(train_ds, batch_size=4, shuffle=True)
    test_dl  = DataLoader(test_ds,  batch_size=4, shuffle=False)

    model = BertForSequenceClassification.from_pretrained(
        'cl-tohoku/bert-base-japanese-whole-word-masking',
        num_labels=num_labels, attn_implementation='eager'
    ).to(device)
    opt = AdamW(model.parameters(), lr=2e-5)

    for ep in range(epochs):
        tr_loss = train_one_epoch(model, train_dl, opt)
        va_loss, acc, f1 = evaluate_model(model, test_dl)
        print(f'  Epoch {ep+1}: train_loss={tr_loss:.4f}, test_loss={va_loss:.4f}, acc={acc:.4f}, f1={f1:.4f}')

    per_branch_bert[bid] = {
        'model': model,
        'tokenizer': tokenizer,
        'label_encoder': le,
        'test_patient_ids': id_test.tolist(),
        'train_patient_ids': id_train.tolist(),
    }

## 4.2. yaml の「分岐 → 選択肢」サイクルで順次推論

`branch_table` を頭から順に走査。各分岐で対応BERTを呼び、softmax 信頼度が閾値以上なら採用、未満なら R3（=不明）で安全側に終端。
「いいえ（=次へ）」が選ばれた場合のみ次の分岐BERTを呼ぶ。

In [ ]:
import torch.nn.functional as F

threshold_confidence = 0.5  # softmax 信頼度の閾値

def sequential_predict(patient_id, df_pairs, branch_table, per_branch_bert,
                       threshold_confidence=0.5):
    # yaml の分岐→選択肢サイクルに従って順次推論し、最終 triage を返す。
    trace = []
    for b in branch_table:
        bid = b['id']
        adopted_col = f'採用ペア_{bid}'
        patient_pairs = df_pairs[(df_pairs['id'] == patient_id) & (df_pairs[adopted_col] == True)]
        input_text = ' '.join(patient_pairs['ペア'].tolist())

        bundle = per_branch_bert[bid]
        model, tokenizer, le = bundle['model'], bundle['tokenizer'], bundle['label_encoder']
        model.eval()

        enc = tokenizer([input_text], padding='max_length', truncation=True,
                        max_length=max_len, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1)[0].cpu().numpy()
        pred_idx_in_encoded = int(probs.argmax())
        confidence = float(probs[pred_idx_in_encoded])
        decoded_label = int(le.inverse_transform([pred_idx_in_encoded])[0])

        if confidence < threshold_confidence:
            # 信頼度不足 → 安全側で R3
            trace.append({'branch': bid, 'predicted_label': decoded_label,
                          'confidence': confidence, 'action': 'low_confidence',
                          'triage': 'R3'})
            return 'R3', trace

        choice_code = LABEL_TO_CHOICE_CODE[decoded_label]
        choice = next(c for c in b['choices'] if c['code'] == choice_code)

        trace.append({'branch': bid, 'predicted_label': decoded_label,
                      'confidence': confidence, 'choice': choice['text'],
                      'action': choice['action']})

        if choice['action'] in ('terminal', 'fallback'):
            return choice['triage'], trace
        # action == 'next' なら次の分岐BERTへ
    # 全部 'next' で抜けた場合 → fallback
    return fallback_triage, trace


# 全患者で推論し結果を表
all_patient_ids = sorted(df_pairs['id'].unique())
seq_results = []
triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}  # データのトリアージ値 0/1/2 → 文字列
for pid in all_patient_ids:
    pred_triage, trace = sequential_predict(pid, df_pairs, branch_table, per_branch_bert,
                                            threshold_confidence=threshold_confidence)
    true_triage_code = df_pairs[df_pairs['id'] == pid]['トリアージ'].iloc[0]
    seq_results.append({
        'id': pid,
        '真のトリアージ': triage_decode.get(int(true_triage_code), str(true_triage_code)),
        'モデル1予測': pred_triage,
        'trace': trace,
    })

seq_df = pd.DataFrame(seq_results)
display(seq_df[['id', '真のトリアージ', 'モデル1予測']])

# 詳細トレースの表示
for r in seq_results:
    print(f'\n[id={r["id"]}] 真={r["真のトリアージ"]} / 予測={r["モデル1予測"]}')
    for step in r['trace']:
        print('   ', step)

# 5. モデル2：選択強化BERT（単一 MultipleChoice BERT）

1つの BERT が、yaml の任意の分岐ノードについて
- **文脈**: 分岐質問 ＋ その分岐に紐づく採用ペア
- **選択肢**: yaml の choices の各テキスト

を入力に取り、選択肢インデックスを予測する。
推論時は branch_table を順に走査し、毎回**同じモデル**で予測 → 選択肢に紐づく action（terminal/next/fallback）で遷移する。

> BERTは「今どこにいて、次にどこに行かせるか」を、入力の **文脈 = 分岐質問** と **選択肢一覧** で認識し、出力した選択肢の `action`（yaml）で次の分岐 or 終端 triage が決まる。

## 5.1. 学習データの生成（患者 × 分岐 のペア）

In [ ]:
mc_examples = []  # {'patient_id', 'branch_id', 'context', 'choices', 'label'}

for b in branch_table:
    bid = b['id']
    label_col = branch_to_label_col[bid]
    adopted_col = f'採用ペア_{bid}'
    choice_texts = [c['text'] for c in b['choices']]
    code_to_idx = {c['code']: i for i, c in enumerate(b['choices'])}

    for pid, sub in df_pairs.groupby('id'):
        adopted_pairs = sub[sub[adopted_col] == True]['ペア'].tolist()
        # 文脈 = 「分岐質問」+ 「採用ペア結合」
        context = b['question'] + ' ' + ' '.join(adopted_pairs)
        gold_data_label = int(sub[label_col].iloc[0])
        gold_code = LABEL_TO_CHOICE_CODE[gold_data_label]
        gold_idx = code_to_idx[gold_code]
        mc_examples.append({
            'patient_id': pid,
            'branch_id': bid,
            'context': context,
            'choices': choice_texts,
            'label': gold_idx,
        })

mc_df = pd.DataFrame(mc_examples)
print(f'MultipleChoice 例数 = (患者数 × 分岐数) = {len(mc_df)}')
display(mc_df.head(10))

## 5.2. BertForMultipleChoice の学習

In [ ]:
from transformers import BertForMultipleChoice

# 患者IDで train/test 分割（同じ患者は train か test のどちらか一方のみ）
train_ids, test_ids = train_test_split(all_patient_ids, test_size=1/3, random_state=42)
mc_train = mc_df[mc_df['patient_id'].isin(train_ids)].reset_index(drop=True)
mc_test  = mc_df[mc_df['patient_id'].isin(test_ids)].reset_index(drop=True)
print(f'mc_train={len(mc_train)}, mc_test={len(mc_test)}')

NUM_CHOICES = max(len(b['choices']) for b in branch_table)
print(f'NUM_CHOICES = {NUM_CHOICES}')

tokenizer_mc = BertJapaneseTokenizer.from_pretrained('cl-tohoku/bert-base-japanese-whole-word-masking')


def encode_mc_batch(df_split, tokenizer, max_len=128):
    # BertForMultipleChoice 用に [batch, num_choices, seq_len] にエンコード
    input_ids_all, attn_all, labels = [], [], []
    for _, row in df_split.iterrows():
        ctx = row['context']
        choices = list(row['choices'])
        while len(choices) < NUM_CHOICES:
            choices.append('')
        enc = tokenizer(
            [ctx] * NUM_CHOICES, choices,
            padding='max_length', truncation=True, max_length=max_len, return_tensors='pt'
        )
        input_ids_all.append(enc['input_ids'])
        attn_all.append(enc['attention_mask'])
        labels.append(row['label'])
    input_ids_all = torch.stack(input_ids_all)   # [B, C, L]
    attn_all = torch.stack(attn_all)             # [B, C, L]
    labels = torch.tensor(labels)
    return input_ids_all, attn_all, labels


train_ids_t, train_attn_t, train_lab_t = encode_mc_batch(mc_train, tokenizer_mc, max_len)
test_ids_t,  test_attn_t,  test_lab_t  = encode_mc_batch(mc_test,  tokenizer_mc, max_len)
print('train shape:', train_ids_t.shape, 'test shape:', test_ids_t.shape)

train_dl_mc = DataLoader(TensorDataset(train_ids_t, train_attn_t, train_lab_t),
                         batch_size=4, shuffle=True)
test_dl_mc  = DataLoader(TensorDataset(test_ids_t,  test_attn_t,  test_lab_t),
                         batch_size=4, shuffle=False)

model_mc = BertForMultipleChoice.from_pretrained(
    'cl-tohoku/bert-base-japanese-whole-word-masking', attn_implementation='eager'
).to(device)
optimizer_mc = AdamW(model_mc.parameters(), lr=2e-5)


def train_mc(model, dataloader, optimizer):
    model.train()
    total = 0
    for ids, mask, lab in dataloader:
        ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, labels=lab)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += out.loss.item()
    return total / len(dataloader)

def eval_mc(model, dataloader):
    model.eval()
    val_loss, preds, gold = 0, [], []
    with torch.no_grad():
        for ids, mask, lab in dataloader:
            ids, mask, lab = ids.to(device), mask.to(device), lab.to(device)
            out = model(input_ids=ids, attention_mask=mask, labels=lab)
            val_loss += out.loss.item()
            preds.extend(out.logits.argmax(dim=-1).cpu().numpy().tolist())
            gold.extend(lab.cpu().numpy().tolist())
    return (val_loss / len(dataloader),
            accuracy_score(gold, preds),
            f1_score(gold, preds, average='weighted', zero_division=0))


for ep in range(epochs):
    tr_loss = train_mc(model_mc, train_dl_mc, optimizer_mc)
    va_loss, acc, f1 = eval_mc(model_mc, test_dl_mc)
    print(f'[MC-BERT] Epoch {ep+1}: train_loss={tr_loss:.4f}, test_loss={va_loss:.4f}, acc={acc:.4f}, f1={f1:.4f}')

## 5.3. yaml の「分岐 → 選択肢」サイクルで遷移図をたどる

In [ ]:
def mc_predict_one(model, tokenizer, branch, patient_pairs_texts, max_len=128):
    # 1分岐に対し MC-BERT で選択肢インデックスと信頼度を返す
    ctx = branch['question'] + ' ' + ' '.join(patient_pairs_texts)
    choices = [c['text'] for c in branch['choices']]
    n_valid = len(choices)
    while len(choices) < NUM_CHOICES:
        choices.append('')
    enc = tokenizer(
        [ctx] * NUM_CHOICES, choices,
        padding='max_length', truncation=True, max_length=max_len, return_tensors='pt'
    )
    ids = enc['input_ids'].unsqueeze(0).to(device)
    mask = enc['attention_mask'].unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=mask).logits  # [1, NUM_CHOICES]
    # 有効な選択肢だけで softmax
    valid_logits = logits[0, :n_valid]
    probs = F.softmax(valid_logits, dim=-1).cpu().numpy()
    pred_idx = int(probs.argmax())
    return pred_idx, float(probs[pred_idx])


def mc_sequential_predict(patient_id, df_pairs, branch_table, model_mc, tokenizer_mc,
                          threshold_confidence=0.5):
    trace = []
    for b in branch_table:
        bid = b['id']
        adopted_col = f'採用ペア_{bid}'
        patient_pairs = df_pairs[(df_pairs['id'] == patient_id) & (df_pairs[adopted_col] == True)]['ペア'].tolist()
        pred_idx, conf = mc_predict_one(model_mc, tokenizer_mc, b, patient_pairs)
        if conf < threshold_confidence:
            trace.append({'branch': bid, 'predicted_idx': pred_idx, 'confidence': conf,
                          'action': 'low_confidence', 'triage': 'R3'})
            return 'R3', trace
        choice = b['choices'][pred_idx]
        trace.append({'branch': bid, 'predicted_idx': pred_idx, 'confidence': conf,
                      'choice': choice['text'], 'action': choice['action']})
        if choice['action'] in ('terminal', 'fallback'):
            return choice['triage'], trace
    return fallback_triage, trace


mc_results = []
for pid in all_patient_ids:
    pred_triage, trace = mc_sequential_predict(pid, df_pairs, branch_table, model_mc, tokenizer_mc,
                                               threshold_confidence=threshold_confidence)
    true_triage_code = df_pairs[df_pairs['id'] == pid]['トリアージ'].iloc[0]
    mc_results.append({
        'id': pid,
        '真のトリアージ': triage_decode.get(int(true_triage_code), str(true_triage_code)),
        'モデル2予測': pred_triage,
        'trace': trace,
    })

mc_results_df = pd.DataFrame(mc_results)
display(mc_results_df[['id', '真のトリアージ', 'モデル2予測']])

for r in mc_results:
    print(f'\n[id={r["id"]}] 真={r["真のトリアージ"]} / 予測={r["モデル2予測"]}')
    for step in r['trace']:
        print('   ', step)

# 6. 2モデルの比較

同じ「yaml の 分岐 → 選択肢 サイクル」を、3つの専用BERT（モデル1）と単一MC-BERT（モデル2）でたどった結果を並べる。

In [ ]:
compare_df = seq_df[['id', '真のトリアージ', 'モデル1予測']].merge(
    mc_results_df[['id', 'モデル2予測']], on='id'
)
compare_df['モデル1正解'] = compare_df['真のトリアージ'] == compare_df['モデル1予測']
compare_df['モデル2正解'] = compare_df['真のトリアージ'] == compare_df['モデル2予測']
display(compare_df)

m1_acc = compare_df['モデル1正解'].mean()
m2_acc = compare_df['モデル2正解'].mean()
print(f'\nモデル1（全BERT 順次）      Accuracy: {m1_acc:.4f}')
print(f'モデル2（選択強化BERT 単一）Accuracy: {m2_acc:.4f}')